# 01 Obtain – Studentenwohnheime (Deutschland + Bonn)

Ziel dieses Notebooks:
- Rohdaten laden (DE-Excel + Bonn-GeoJSON)
- Struktur prüfen
- Aus dem DE-Excel eine erste “Obtain”-Tabelle extrahieren und als CSV speichern:
  `data/interim/student_housing_de_obtain.csv`



In [1]:
import pandas as pd
from pathlib import Path
import re

path_xlsx = Path("../../data/raw/student_housing/Studentenwohnheime_DE.xlsx")
path_bonn = Path("../../data/raw/student_housing/Studentenwohnheime_BONN.json")

path_xlsx.exists(), path_xlsx.resolve(), path_bonn.exists(), path_bonn.resolve()


C:\Users\User\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


(True,
 WindowsPath('C:/Users/User/Desktop/DATA INFORMATION SCIENCE/DIS08/dis08-miete-nrw/data/raw/student_housing/Studentenwohnheime_DE.xlsx'),
 True,
 WindowsPath('C:/Users/User/Desktop/DATA INFORMATION SCIENCE/DIS08/dis08-miete-nrw/data/raw/student_housing/Studentenwohnheime_BONN.json'))

In [2]:
xls = pd.ExcelFile(path_xlsx)
xls.sheet_names


['StaGus\xa0-\xa0Auswertung']

In [3]:
sheet = xls.sheet_names[0]

top = pd.read_excel(path_xlsx, sheet_name=sheet, header=None, nrows=20)

year_row_candidates = top.apply(
    lambda r: r.astype(str).str.fullmatch(r"\s*2001\s*", na=False).any(),
    axis=1
)

year_row = int(year_row_candidates[year_row_candidates].index[0])
header_rows = [year_row - 1, year_row, year_row + 1]  

year_row, header_rows, top.iloc[year_row-2:year_row+3, :8]


(5,
 [4, 5, 6],
                    0                      1          2       3          4  \
 3                NaN                    NaN        NaN     NaN        NaN   
 4                NaN  Studierendenwohnheime        NaN     NaN        NaN   
 5                NaN                   2001        NaN    2002        NaN   
 6                NaN                 Anzahl  Plätze 3)  Anzahl  Plätze 3)   
 7  Hochschulort/Land                 Anzahl        NaN     NaN        NaN   
 
         5          6       7  
 3     NaN        NaN     NaN  
 4     NaN        NaN     NaN  
 5    2003        NaN    2004  
 6  Anzahl  Plätze 3)  Anzahl  
 7     NaN        NaN     NaN  )

In [4]:
df_raw = pd.read_excel(
    path_xlsx,
    sheet_name=sheet,
    header=header_rows
)

df_raw.shape, df_raw.head(3)


C:\Users\User\anaconda3\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


((249, 54),
   Unnamed: 0_level_0 Studierendenwohnheime                                    \
   Unnamed: 0_level_1                  2001             2002             2003   
   Unnamed: 0_level_2                Anzahl Plätze 3) Anzahl Plätze 3) Anzahl   
 0  Hochschulort/Land                Anzahl       NaN    NaN       NaN    NaN   
 1              Aalen                     3       251      3       251      3   
 2           Albstadt                     -        20      -        20      -   
 
                                      ...                                    \
               2004             2005  ...      2022   2023             2024   
   Plätze 3) Anzahl Plätze 3) Anzahl  ... Plätze 3) Anzahl Plätze 3) Anzahl   
 0       NaN    NaN       NaN    NaN  ...       NaN    NaN       NaN    NaN   
 1       251      3       251      4  ...       421      6       421      6   
 2        20      2        80      -  ...        82      2        82      2   
 
             Studierende

In [5]:
def flatten_col(col):
    parts = [str(x).strip() for x in col if (x is not None and str(x).strip() != "" and str(x) != "nan")]
    return " | ".join(parts)

df = df_raw.copy()
df.columns = [flatten_col(c) if isinstance(c, tuple) else str(c) for c in df.columns]

df.columns[:10], df.shape


(Index(['Unnamed: 0_level_0 | Unnamed: 0_level_1 | Unnamed: 0_level_2',
        'Studierendenwohnheime | 2001 | Anzahl',
        'Studierendenwohnheime | 2001 | Plätze 3)',
        'Studierendenwohnheime | 2002 | Anzahl',
        'Studierendenwohnheime | 2002 | Plätze 3)',
        'Studierendenwohnheime | 2003 | Anzahl',
        'Studierendenwohnheime | 2003 | Plätze 3)',
        'Studierendenwohnheime | 2004 | Anzahl',
        'Studierendenwohnheime | 2004 | Plätze 3)',
        'Studierendenwohnheime | 2005 | Anzahl'],
       dtype='object'),
 (249, 54))

In [6]:
col_city = df.columns[0]

def has(col, pattern):
    return re.search(pattern, col, flags=re.IGNORECASE) is not None

col_wohnheime_2025 = [c for c in df.columns if has(c, r"\b2025\b") and has(c, r"wohnheime") and has(c, r"anzahl")]
col_plaetze_2025   = [c for c in df.columns if has(c, r"\b2025\b") and has(c, r"plätz|plaetz|plätze|plaetze")]
col_stud_ws        = [c for c in df.columns if has(c, r"WS\s*2024/2025") and has(c, r"insgesamt")]
col_stud_pro_platz = [c for c in df.columns if has(c, r"\b2025\b") and has(c, r"je wohnheimplatz")]

col_city, col_wohnheime_2025[:3], col_plaetze_2025[:3], col_stud_ws[:3], col_stud_pro_platz[:3]


('Unnamed: 0_level_0 | Unnamed: 0_level_1 | Unnamed: 0_level_2',
 ['Studierenden-wohnheime | 2025 | Anzahl'],
 ['Studierenden-wohnheime | 2025 | Plätze 3)'],
 ['Studierende | WS 2024/2025 | Insgesamt 2)'],
 ['Studierende | 2025 | Je Wohnheimplatz'])

In [7]:
use_cols = {
    "hochschulort": col_city,
    "wohnheime_2025_anzahl": col_wohnheime_2025[0],
    "wohnheimplaetze_2025": col_plaetze_2025[0],
    "studierende_ws_2024_2025": col_stud_ws[0],
    "studierende_je_wohnheimplatz_2025": col_stud_pro_platz[0],
}

df_de_obtain = df[list(use_cols.values())].rename(columns={v: k for k, v in use_cols.items()})

df_de_obtain = df_de_obtain[df_de_obtain["hochschulort"].notna()].copy()
df_de_obtain = df_de_obtain[df_de_obtain["hochschulort"].astype(str).str.strip() != "Hochschulort/Land"].copy()

df_de_obtain.head(10), df_de_obtain.shape


(          hochschulort wohnheime_2025_anzahl wohnheimplaetze_2025  \
 1                Aalen                     7                  531   
 2             Albstadt                     2                   82   
 3   Bad Mergentheim 4)                     2                   35   
 4             Biberach                     1                   64   
 5         Esslingen 5)                     5                  857   
 6             Freiburg                    57                 6649   
 7      Friedrichshafen                     1                  197   
 8           Furtwangen                     5                  291   
 9           Geislingen                     4                  244   
 10        Göppingen 5)                     1                  128   
 
    studierende_ws_2024_2025 studierende_je_wohnheimplatz_2025  
 1                      4117                                 8  
 2                      1392                                17  
 3                       550     

In [8]:
out_path = Path("../../data/processed/student_housing_de_obtain.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

df_de_obtain.to_csv(out_path, index=False)

out_path.exists(), out_path.resolve()


(True,
 WindowsPath('C:/Users/User/Desktop/DATA INFORMATION SCIENCE/DIS08/dis08-miete-nrw/data/processed/student_housing_de_obtain.csv'))

In [9]:
import json

with open(path_bonn, "r", encoding="utf-8") as f:
    geo = json.load(f)

features = geo["features"]
props = [feat["properties"] for feat in features]
coords = [feat["geometry"]["coordinates"] for feat in features]

df_bonn = pd.DataFrame(props)
df_bonn["lon"] = [c[0] for c in coords]
df_bonn["lat"] = [c[1] for c in coords]

df_bonn.shape, df_bonn.columns


((58, 22),
 Index(['point_id', 'name', 'ort', 'ort_bezeichnung', 'strasse',
        'strasse_bezeichnung', 'hausnr', 'plz', 'telefon', 'fax', 'email',
        'internet', 'traeger', 'oeffentlich', 'barrierefrei', 'sammelfeld',
        'erfassung_datum', 'strasse_ort', 'plzort', 'adresse', 'lon', 'lat'],
       dtype='object'))